# Udaplay Project
Based on UDAPLAY Starter

In thsi file all the necessary tool and the agent itself is defined!

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# Import the necessary libs
# 
import os
from dotenv import load_dotenv, find_dotenv
import json
from typing import List, Dict, Any, Optional, Literal
from pydantic import BaseModel, Field


import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

from openai import OpenAI

# Import the agent framework and tooling
from lib.agents import Agent
from lib.tooling import tool

In [3]:
# Load Environment, check if all is set properly
load_dotenv(find_dotenv(), override=True)
# assert os.getenv("OPENAI_dummy"), "OPENAI_dummy is not set"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Environment variables are loaded in the setup cell above.

OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY  = os.getenv("TAVILY_API_KEY")

# Initialize clients with error handling
# here i had some issues in the beginnign with

try:
    client = OpenAI(
        base_url=OPENAI_BASE_URL,
        api_key=OPENAI_API_KEY,
    )
except Exception as e:
    raise ValueError(f'Failed to initialize OpenAI client: {str(e)}')
try:
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
except Exception as e:
    raise ValueError(f'Failed to initialize Tavily client: {str(e)}')

print('--> API clients initialized successfully!')


--> API clients initialized successfully!


In [5]:
# define AgentAnswer helper class
# this will ease up in the model usage later in this file

class AgentAnswer(BaseModel):
    answer: str = Field(description="The direct answer to the user's question.")
    source: str = Field(description="The source of the information, either the game's name or 'Web Search'.")
    fallback_used: bool = Field(description="True if a web search was required to answer the question.")
    internal_reasoning: str = Field(description="The agent's step-by-step reasoning for its conclusion.")

#

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# Create retrieve_game tool
# It should use the chroma client and collection you created in Part 1 (same path and name)
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, PlayStation 5, Xbox 360...
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

# ---------- Vector store ----------
# Re-open the persistent collection populated in Part 1. We do NOT re-ingest
# here so this notebook can be run independently of Part 1, as long as the
# `./chromadb` folder already contains the collection.

# Tool No 1: RAG retrieval

@tool
def retrieve_game_info(query: str, n_results: int = 3) -> list[str]:
    """
    Queries the ChromaDB vector store to find relevant game information.
    """
    print(f"Tool: Retrieving game info for query: '{query}'")
    chroma_client = chromadb.PersistentClient(path="chromadb")

    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        api_base="https://openai.vocareum.com/v1",  # For Vocareum
        model_name="text-embedding-ada-002"
    )

    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

    # get the collection first
    results = collection.query(query_texts=[query], n_results=n_results)

    # do some formatting before return the results (--> retrieved_docs)
    retrieved_docs = []
    if results and results['metadatas'][0]:
        for meta in results['metadatas'][0]:
            # Pass the full JSON metadata as a string
            retrieved_docs.append(json.dumps(meta))

    return retrieved_docs
    

#### Evaluate Retrieval Tool

In [7]:
# Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

# Tool No 2: evaluate_retrieval

@tool
def evaluate_retrieval(query: str, context: list[str]) -> bool:
    """
    Uses an LLM to decide if the retrieved context is sufficient to answer the query.
    """
    # ... (rest of the function is the same) ...

    # A simple but effective prompt for the LLM
    prompt = f"""
    Based *only* on the provided context below, can you confidently answer the following user question?
    Respond with only "yes" or "no".

    User Question: "{query}"

    Context:
    ---
    {context}
    ---
    """

    # Use the configured client object here
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0.0
    )

    decision = response.choices[0].message.content.strip().lower()
    print(f"Evaluation: LLM decision is '{decision}'.")
    return "yes" in decision

#### Game Web Search Tool

In [8]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.

# Tool No 3: game_web_reserch tool

@tool
def game_web_search(query: str) -> str:
    """
    searches the web with Tavily when the vector DB results aren't good enough.
    """
    print(f"Tool: Performing web search for query: '{query}'")
    try:
        response = tavily_client.search(query=query, search_depth="basic")
        # We'll return the most relevant search result content
        return response['results'][0]['content']
    except Exception as e:
        print(f"Error during Tavily search: {e}")
        return "Web search failed."

### StateMachine
Implementation of the StateMachine and helper functions

In [22]:
from typing import TypedDict, List, Optional, Union
from lib.state_machine import StateMachine,Step, EntryPoint, Termination, Run
from lib.llm import LLM
from lib.messages import AIMessage, UserMessage, SystemMessage, ToolMessage
from lib.tooling import Tool, ToolCall, tool

tools = [game_web_search, evaluate_retrieval, retrieve_game_info]

class AgentState(TypedDict):
    user_query: str
    instructions: str
    messages: List[dict]
    current_tool_calls: Optional[List[ToolCall]]

class Agent:
    def __init__(self, 
                 model_name: str,
                 instructions: str, 
                 tools: List[Tool] = None,
                 temperature: float = 0.7):
        """
        Initialize an Agent instance
        
        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature
        print("======================= BEGIN DUMP ==============================")
        print(f"Tools {self.tools}")
        print(f"Model {self.model_name}")
        print("======================= END DUMP ================================")
                
        # Initialize state machine
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        """Step logic: Prepare messages for LLM consumption"""

        messages = [
            SystemMessage(content=state["instructions"]),
            UserMessage(content=state["user_query"])
        ]
        
        return {
            "messages": messages
        }

    def _llm_step(self, state: AgentState) -> AgentState:
        """Step logic: Process the current state through the LLM"""

        # Initialize LLM
        llm = LLM(
            model=self.model_name,
            temperature=self.temperature,
            tools=self.tools
        )

        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls if response.tool_calls else None

        # Create AI message with content and tool calls
        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)
        
        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls
        }

    def _tool_step(self, state: AgentState) -> AgentState:
        """Step logic: Execute any pending tool calls"""
        tool_calls = state["current_tool_calls"] or []
        tool_messages = []
        
        for call in tool_calls:
            # Access tool call data correctly
            function_name = call.function.name
            function_args = json.loads(call.function.arguments)
            tool_call_id = call.id
            # Find the matching tool
            tool = next((t for t in self.tools if t.name == function_name), None)
            if tool:
                result = tool(**function_args)
                tool_messages.append(
                    ToolMessage(
                        content=json.dumps(result), 
                        tool_call_id=tool_call_id, 
                        name=function_name, 
                    )
                )
        
        # Clear tool calls and add results to messages
        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None
        }

    def _create_state_machine(self) -> StateMachine[AgentState]:
        """Create the internal state machine for the agent"""
        machine = StateMachine[AgentState](AgentState)
        
        # Create steps
        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[AgentState]("llm_processor", self._llm_step)
        tool_executor = Step[AgentState]("tool_executor", self._tool_step)
        termination = Termination[AgentState]()
        
        machine.add_steps([entry, message_prep, llm_processor, tool_executor, termination])
        
        # Add transitions
        machine.connect(entry, message_prep)
        machine.connect(message_prep, llm_processor)
        
        # Transition based on whether there are tool calls
        def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
            """Transition logic: Check if there are tool calls"""
            if state.get("current_tool_calls"):
                return tool_executor
            return termination
        
        machine.connect(llm_processor, [tool_executor, termination], check_tool_calls)
        machine.connect(tool_executor, llm_processor)  # Go back to llm after tool execution
        
        return machine

    def invoke(self, query: str) -> Run:
        """
        Run the agent on a query
        
        Args:
            query: The user's query to process
            
        Returns:
            The final run object after processing
        """

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages":  [],
        }

        run_object = self.workflow.run(initial_state)

        return run_object




### Agent with State Machine
First create instance of State Machien enabled agent

In [23]:
UdaPlay_agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=(
        """You are UdaPlay, an AI research agent specialized in the video game industry.
        Your capabilities include:
        1. Searching a comprehensive database of video games using semantic search
        2. Evaluating whether retrieved information is sufficient to answer queries
        3. Performing web searches when local knowledge is insufficient
        When answering questions:
        1. First, search your local database for relevant game information
        2. Evaluate if the retrieved context is sufficient to answer the question
        3. If not sufficient, perform a web search for additional information
        4. Provide comprehensive, accurate answers based on available information
        5. Only show facts from database or the web, never invent information

        Always be friendly, helpful, accurate, and cite your sources when possible.
        For queries about game releases, platforms, genres, or specific game details,
        prioritize information from your local database first.

        Always structure your final response as valid JSON using the AgentAnswer format:
        {
            "answer": "The direct answer to the user's question",
            "source": "Source of information (game name or 'Web Search')",
            "fallback_used": true/false,
            "internal_reasoning": "Step-by-step reasoning process
        }

        Never try to respond directly if the question needs a tool. 
        But if you don't have a tool available, you can respond directly. 
        f"The actions you have are the Tools: {tools}. \n
        """
    )
)

# Test Code
#run_object = UdaPlay_agent.invoke(
#    query="Tell me about Nintendo games in our database",
#)
#
#run_object.get_final_state()["messages"]



======================= BEGIN DUMP ==============================
Tools [<Tool name=game_web_search params=['query']>, <Tool name=evaluate_retrieval params=['query', 'context']>, <Tool name=retrieve_game_info params=['query', 'n_results']>]
Model gpt-4o-mini
======================= END DUMP ================================


Finally execute the Agent!

In [24]:
# Here are the provided test questions
# "Tell me about Nintendo games in our database",
# "When was Pokémon Gold and Silver released?",
# "What is Rockstar Games working on right now?",
list_of_queries = [ 
                    "Who published Grand Theft Auto: San Andreas?", 
                    "When was it released?",
                    "What platform was Super Mario 64 launched on?",
                    "Which other successfull games have been released on this platform?", 
                    
]

# reserve some space for sessions: limited to 1000 currently, production usage would require dynamic management
run = [0] * 1000
state = [0] * 1000

# Loop along the test questions, execute the agent and dumpt the results
i = 0
for query in list_of_queries:
    i=i+1
    print("----------------------------------------------------------------------")
    print(f"Query {i}: {query}")
    run[i] = UdaPlay_agent.invoke(query)
    state[i] = run[i].get_final_state()
    print(f"Answer {i}: {state[i]['messages'][-1].content}")
    print(f"Messages in session: {len(state[i]['messages'])}")
    print("----------------------------------------------------------------------")




----------------------------------------------------------------------
Query 1: Who published Grand Theft Auto: San Andreas?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: llm_processor
Tool: Retrieving game info for query: 'Grand Theft Auto: San Andreas'
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer 1: {
    "answer": "Grand Theft Auto: San Andreas was published by Rockstar Games.",
    "source": "Grand Theft Auto: San Andreas",
    "fallback_used": false,
    "internal_reasoning": "I retrieved information about Grand Theft Auto: San Andreas from the database, which included the publisher's name."
}
Messages in session: 5
----------------------------------------------------------------------
----------------------------------------------------------------------
Query 2: When was it released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer 2: {
    "answer": "Could you please specify which game you are ask

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: llm_processor
Tool: Retrieving game info for query: 'Super Mario 64'
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer 3: {
    "answer": "Super Mario 64 was launched on the Nintendo 64.",
    "source": "Super Mario 64",
    "fallback_used": false,
    "internal_reasoning": "I retrieved relevant information about Super Mario 64 from the database, which confirmed that it was launched on the Nintendo 64."
}
Messages in session: 5
----------------------------------------------------------------------
----------------------------------------------------------------------
Query 4: Which other successfull games have been released on this platform?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: llm_processor
Tool: Retrieving game info for query: 'successful games released on this platform'
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluation: LLM decision is 'no'.
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Tool: Performing web search for query: 'successful games released on Nintendo 64'
Error during Tavily search: Unauthorized: missing or invalid API key.
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Tool: Performing web search for query: 'successful games released on PlayStation 1'
Error during Tavily search: Unauthorized: missing or invalid API key.
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Tool: Performing web search for query: 'successful games released on Super Nintendo Entertainment System (SNES)'
Error during Tavily search: Unauthorized: missing or invali

### (Optional) Advanced — Stand Out

In [12]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes